# 68 - Cross-Dataset Early Fusion → Primer Test (Skema 2 Extension)

Melengkapi Skema 2 dengan **Early Fusion** yang sebelumnya terlewat di nb 63 (cuma 6 model standard). **Inference-only** — reuse checkpoint Early Fusion dari nb 66 yang sudah dijalankan di VPS (commit `100cd95`).

**Strategi:**
1. Load Early Fusion model yang sudah di-train di source dataset (dari nb 66)
2. Generate 4-channel input untuk Primer test (image + heatmap)
3. Inference di Primer test (929 images) → hitung Macro/Micro/Weighted F1 + Accuracy
4. Tidak ada training baru — super cepat (~5-10 min total)

**Matriks:** 4 source (CK+/JAFFE/RAF-DB/KDEF) × 2 class (7c/4c) × 2 backbone (EF scratch + EF TL) = **16 inference runs**

**Hipotesis:** Heatmap landmark (geometric info) lebih **domain-invariant** daripada RGB image features → Early Fusion mungkin unggul di cross-dataset transfer dibanding CNN single-modality.

**Output:**
- Update `models/benchmark/crossdataset/cross_{source}_{num}c.json` dengan key `EarlyFusion_B1` dan `EarlyFusion_TL_B1`
- Update `models/benchmark/crossdataset/all_cross_results.json` combined

**Prerequisites:**
1. Checkpoint Early Fusion dari nb 66 sudah ada di `models/benchmark/{ds}/...` ✓ (commit `100cd95`)
2. Primer conf60 heatmap ada: `data/dataset_frontonly_conf60/X_test_heatmaps.npy`

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionEarlyFusion, EmotionEarlyFusionTransfer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 64  # inference only, bisa lebih besar dari training

MODELS_DIR = PROJECT_ROOT / 'models' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
CROSS_DIR = MODELS_DIR / 'crossdataset'
CROSS_DIR.mkdir(parents=True, exist_ok=True)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

Device: cuda
Setup complete.


In [2]:
# ── Load Primer test set as 4-channel ──

primer_img = np.load(PRIMER_DIR / 'X_test_images.npy')       # (N, 224, 224, 3)
primer_heat = np.load(PRIMER_DIR / 'X_test_heatmaps.npy')    # (N, 224, 224)
primer_y7 = np.load(PRIMER_DIR / 'y_test.npy')               # (N,)

if primer_heat.ndim == 3:
    primer_heat = primer_heat[..., None]
primer_x4 = np.concatenate([primer_img, primer_heat], axis=-1).astype(np.float32, copy=False)
primer_y4 = REMAP_4[primer_y7]

print(f'Primer test 4-ch: {primer_x4.shape}  y_test_7c: {primer_y7.shape}  y_test_4c: {primer_y4.shape}')
print(f'7-class dist: {np.bincount(primer_y7, minlength=7).tolist()}')
print(f'4-class dist: {np.bincount(primer_y4, minlength=4).tolist()}')

Primer test 4-ch: (929, 224, 224, 4)  y_test_7c: (929,)  y_test_4c: (929,)
7-class dist: [688, 183, 50, 2, 1, 2, 3]
4-class dist: [688, 183, 50, 8]


In [3]:
# ── Helpers ──

def make_loader_4ch(x4, y, batch_size=BATCH_SIZE):
    t = torch.from_numpy(x4).permute(0, 3, 1, 2).contiguous()  # (N, 4, 224, 224)
    ys = torch.from_numpy(y).long()
    ds = TensorDataset(t, ys)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)


def checkpoint_dir(dataset_name, num_classes, model_key):
    """Same convention as nb 66.
    - ckplus/jaffe: models/benchmark/{ds}/{ds}_{num}c/{key}/
    - rafdb/kdef:   models/benchmark/{ds}/{num}c/{key}/
    """
    if dataset_name in ('ckplus', 'jaffe'):
        return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c' / model_key
    return MODELS_DIR / dataset_name / f'{num_classes}c' / model_key


def metrics_triple(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }


@torch.no_grad()
def inference_primer(model_class, num_classes, checkpoint_path, x4, y):
    model = model_class(num_classes=num_classes).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()
    loader = make_loader_4ch(x4, y)
    preds = []
    for xb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        preds.append(model(xb).argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(preds)
    return metrics_triple(y, y_pred)


def cross_early_fusion(dataset_name, num_classes):
    print(f"\n{'='*70}")
    print(f'  Cross: {dataset_name.upper()} {num_classes}c -> Primer test')
    print(f"{'='*70}")

    # Select Primer labels matching model output size
    y_primer = primer_y7 if num_classes == 7 else primer_y4

    # Load existing cross_{ds}_{c}.json (kalau ada), append new results
    cross_file = CROSS_DIR / f'cross_{dataset_name}_{num_classes}c.json'
    if cross_file.exists():
        with open(cross_file) as f:
            existing = json.load(f)
    else:
        existing = {}

    out = {}
    configs = [
        ('EarlyFusion_B1', EmotionEarlyFusion),
        ('EarlyFusion_TL_B1', EmotionEarlyFusionTransfer),
    ]
    for key, cls in configs:
        ckpt_path = checkpoint_dir(dataset_name, num_classes, key) / 'model.pth'
        if not ckpt_path.exists():
            print(f'  [SKIP] {key}: checkpoint not found at {ckpt_path}')
            continue
        r = inference_primer(cls, num_classes, ckpt_path, primer_x4, y_primer)
        out[key] = r
        existing[key] = r
        print(f"    {key:<22} Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  "
              f"Weighted={r['weighted_f1']:.4f}  Acc={r['accuracy']:.4f}")

    # Save to per-source JSON
    with open(cross_file, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'  Updated: {cross_file.name}')
    return out


print('Helpers ready.')

Helpers ready.


## Run Cross-Dataset Early Fusion (4 datasets × 2 classes = 8 combos)

In [4]:
all_results = {}

# CK+
all_results['ckplus_7c'] = cross_early_fusion('ckplus', 7)
all_results['ckplus_4c'] = cross_early_fusion('ckplus', 4)

# JAFFE
all_results['jaffe_7c'] = cross_early_fusion('jaffe', 7)
all_results['jaffe_4c'] = cross_early_fusion('jaffe', 4)

# RAF-DB
all_results['rafdb_7c'] = cross_early_fusion('rafdb', 7)
all_results['rafdb_4c'] = cross_early_fusion('rafdb', 4)

# KDEF
all_results['kdef_7c'] = cross_early_fusion('kdef', 7)
all_results['kdef_4c'] = cross_early_fusion('kdef', 4)


  Cross: CKPLUS 7c -> Primer test


    EarlyFusion_B1         Macro=0.1254  Micro=0.6857  Weighted=0.6378  Acc=0.6857


    EarlyFusion_TL_B1      Macro=0.1792  Micro=0.7051  Weighted=0.6764  Acc=0.7051
  Updated: cross_ckplus_7c.json

  Cross: CKPLUS 4c -> Primer test


    EarlyFusion_B1         Macro=0.2072  Micro=0.5770  Weighted=0.5819  Acc=0.5770


    EarlyFusion_TL_B1      Macro=0.1012  Micro=0.1938  Weighted=0.2878  Acc=0.1938
  Updated: cross_ckplus_4c.json

  Cross: JAFFE 7c -> Primer test


...

    EarlyFusion_TL_B1      Macro=0.3115  Micro=0.4769  Weighted=0.5754  Acc=0.4769
  Updated: cross_rafdb_4c.json

  Cross: KDEF 7c -> Primer test


    EarlyFusion_B1         Macro=0.0073  Micro=0.0097  Weighted=0.0106  Acc=0.0097


    EarlyFusion_TL_B1      Macro=0.0292  Micro=0.0538  Weighted=0.0972  Acc=0.0538
  Updated: cross_kdef_7c.json

  Cross: KDEF 4c -> Primer test


    EarlyFusion_B1         Macro=0.0387  Micro=0.0398  Weighted=0.0235  Acc=0.0398


    EarlyFusion_TL_B1      Macro=0.1037  Micro=0.1012  Weighted=0.1518  Acc=0.1012
  Updated: cross_kdef_4c.json


In [5]:
# ── Update combined all_cross_results.json ──

combined_file = CROSS_DIR / 'all_cross_results.json'
if combined_file.exists():
    with open(combined_file) as f:
        combined = json.load(f)
else:
    combined = {}

for src_key, ef_results in all_results.items():
    if src_key not in combined:
        combined[src_key] = {}
    combined[src_key].update(ef_results)

with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2)
print(f'Updated combined file: {combined_file.name}')

Updated combined file: all_cross_results.json


## Ringkasan Cross-Dataset Early Fusion → Primer

In [6]:
print(f"\n{'='*88}")
print(f'  Cross-Dataset Early Fusion -> Primer conf60 test (929 imgs)')
print(f"{'='*88}")
print(f"  {'Source':<14} {'Config':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Acc':>10}")
print(f"  {'-'*82}")
for src_key, res in all_results.items():
    for cfg, r in res.items():
        print(f"  {src_key:<14} {cfg:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} "
              f"{r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")

# Compare with non-EF cross-dataset (from existing all_cross_results.json)
print(f"\nExisting (non-EF) cross-dataset best Macro F1 per source (reference):")
for src in ['ckplus_7c','ckplus_4c','jaffe_7c','jaffe_4c','rafdb_7c','rafdb_4c','kdef_7c','kdef_4c']:
    if src in combined:
        non_ef = {k: v for k, v in combined[src].items() if 'EarlyFusion' not in k}
        if non_ef:
            best_k = max(non_ef.keys(), key=lambda k: non_ef[k].get('macro_f1', 0))
            print(f"  {src:<14} best non-EF: {best_k:<22} Macro={non_ef[best_k]['macro_f1']:.4f}")


  Cross-Dataset Early Fusion -> Primer conf60 test (929 imgs)
  Source         Config                      Macro      Micro   Weighted        Acc
  ----------------------------------------------------------------------------------
  ckplus_7c      EarlyFusion_B1             0.1254     0.6857     0.6378     0.6857
  ckplus_7c      EarlyFusion_TL_B1          0.1792     0.7051     0.6764     0.7051
  ckplus_4c      EarlyFusion_B1             0.2072     0.5770     0.5819     0.5770
  ckplus_4c      EarlyFusion_TL_B1          0.1012     0.1938     0.2878     0.1938
  jaffe_7c       EarlyFusion_B1             0.0264     0.0323     0.0257     0.0323
  jaffe_7c       EarlyFusion_TL_B1          0.0006     0.0022     0.0000     0.0022
  jaffe_4c       EarlyFusion_B1             0.0043     0.0086     0.0001     0.0086
  jaffe_4c       EarlyFusion_TL_B1          0.0043     0.0086     0.0001     0.0086
  rafdb_7c       EarlyFusion_B1             0.1424     0.3649     0.4763     0.3649
  rafdb_7c  